# Snippet from Cookbook.md


In [ ]:
#!/usr/bin/env python3
import json
import glob
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
def build_portfolio(student_name, trace_dir, output_dir='portfolios'):
    output_path = Path(output_dir) / student_name
    output_path.mkdir(parents=True, exist_ok=True)
    traces = []
    for trace_file in sorted(glob.glob(f"{trace_dir}/{student_name}_*.json")):
        with open(trace_file) as f:
            trace = json.load(f)
            trace['filename'] = Path(trace_file).name
            traces.append(trace)
    if not traces:
        print(f"No traces for {student_name}")
        return
    html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>{student_name} - Learning Portfolio</title>
    <style>
        body {{ font-family: 'Segoe UI', sans-serif; margin: 40px; background: #f5f5f5; }}
        .header {{ background: #1976d2; color: white; padding: 20px; border-radius: 8px; }}
        .trace {{ background: white; margin: 20px 0; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }}
        .metrics {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin: 15px 0; }}
        .metric {{ background: #e3f2fd; padding: 10px; border-radius: 4px; text-align: center; }}
        .metric-value {{ font-size: 24px; font-weight: bold; color: #1976d2; }}
        .metric-label {{ font-size: 12px; color: #666; }}
        .reflection {{ background: #fff9c4; padding: 15px; border-left: 4px solid #f57c00; margin: 15px 0; }}
        .good {{ color: #2e7d32; }}
        .warning {{ color: #f57c00; }}
        .error {{ color: #c62828; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>{student_name}'s Learning Journey</h1>
        <p>Compitum Portfolio • {datetime.now().strftime('%B %d, %Y')}</p>
        <p>Total Routes: {len(traces)}</p>
    </div>
"""
    avg_drift = sum(t['drift']['drift_ema'] for t in traces) / len(traces)
    avg_entropy = sum(t['boundary']['entropy'] for t in traces) / len(traces)
    avg_utility = sum(t['utility'] for t in traces) / len(traces)
    html += f"""
        <div class="trace">
            <h2>Portfolio Summary</h2>
            <div class="metrics">
                <div class="metric">
                    <div class="metric-value {'good' if avg_drift < 0 else 'error'}">{avg_drift:.3f}</div>
                    <div class="metric-label">Avg Drift</div>
                </div>
                <div class="metric">
                    <div class="metric-value {'good' if avg_entropy > 1.0 else 'warning'}">{avg_entropy:.2f}</div>
                    <div class="metric-label">Avg Entropy</div>
                </div>
                <div class="metric">
                    <div class="metric-value">{avg_utility:.2f}</div>
                    <div class="metric-label">Avg Utility</div>
                </div>
                <div class="metric">
                    <div class="metric-value">{len(traces)}</div>
                    <div class="metric-label">Routes</div>
                </div>
            </div>
            <div class="reflection">
                <strong>Growth Notes:</strong><br/>
                {'Consistent descent' if avg_drift < -0.01 else 'Inconsistent drift—tune constraints'}
                <br/>
                {'Diverse exploration' if avg_entropy > 1.5 else 'Low diversity—vary prompts'}
            </div>
        </div>
    """
    for i, trace in enumerate(traces, 1):
        html += f"""
        <div class="trace">
            <h3>Route {i}: {trace['prompt'][:80]}...</h3>
            <p><em>Model: {trace['model']}</em></p>
            <div class="metrics">
                <div class="metric">
                    <div class="metric-value {'good' if trace['drift']['drift_ema'] < 0 else 'error'}">
                        {trace['drift']['drift_ema']:.3f}
                    </div>
                    <div class="metric-label">Drift</div>
                </div>
                <div class="metric">
                    <div class="metric-value">{trace['boundary']['entropy']:.2f}</div>
                    <div class="metric-label">Entropy</div>
                </div>
                <div class="metric">
                    <div class="metric-value">{trace['utility']:.2f}</div>
                    <div class="metric-label">Utility</div>
                </div>
                <div class="metric">
                    <div class="metric-value">{'Yes' if trace['constraints']['feasible'] else 'No'}</div>
                    <div class="metric-label">Feasible</div>
                </div>
            </div>
            <h4>Constraints:</h4>
            <ul>
        """
        for name, data in trace['constraints'].items():
            status = 'Yes' if data['slack'] >= 0 else 'No'
            html += f"<li>{status} {name}: {data['actual']:.3f} / {data['bound']:.3f}</li>"
        html += "</ul><h4>Shadow Prices:</h4><ul>"
        for name, value in trace['shadow_prices'].items():
            level = 'error' if value > 0.5 else 'warning' if value > 0.1 else 'good'
            html += f"<li class='{level}'>{name}: λ={value:.2f}</li>"
        html += f"""
            </ul>
            <div class="reflection">
                <strong>Reflection Prompt:</strong>
                What does this certificate indicate about prompt complexity?
                Why high or low shadow prices for constraints?
            </div>
        </div>
        """
    html += """
        <div class="trace">
            <h2>Next Steps</h2>
            <ul>
                <li>Adjust constraint bounds</li>
                <li>Test boundary prompts</li>
                <li>Compare with peers</li>
                <li>Reflect on patterns</li>
            </ul>
        </div>
    </body>
    </html>
    """
    portfolio_file = output_path / 'portfolio.html'
    with open(portfolio_file, 'w') as f:
        f.write(html)
    print(f"Portfolio: {portfolio_file}")
    plot_student_progress(traces, output_path)
def plot_student_progress(traces, output_dir):
    fig, axes = plt.subplots(2, 1, figsize=(10, 8))
    steps = list(range(1, len(traces) + 1))
    drifts = [t['drift']['drift_ema'] for t in traces]
    entropies = [t['boundary']['entropy'] for t in traces]
    axes[0].plot(steps, drifts, 'o-', color='green', linewidth=2)
    axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    axes[0].set_ylabel('Drift Signal')
    axes[0].set_title('Convergence Over Routes')
    axes[0].grid(alpha=0.3)
    axes[1].plot(steps, entropies, 's-', color='blue', linewidth=2)
    axes[1].axhline(y=1.0, color='orange', linestyle='--', alpha=0.5)
    axes[1].set_ylabel('Entropy')
    axes[1].set_xlabel('Route Number')
    axes[1].set_title('Diversity Over Time')
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_dir / 'progress.png', dpi=150)
    print(f"Chart: {output_dir / 'progress.png'}")
if __name__ == "__main__":
    import sys
    if len(sys.argv) < 3:
        print("Usage: python build_student_portfolio.py <student_name> <trace_directory>")
        sys.exit(1)
    build_portfolio(sys.argv[1], sys.argv[2])
